# Audio Source Tracking System — soqv1
### Sawyer Falkenbush, Omar Mohammed, Quintin Hatzis
#### ELE 392 – DSP and Control Systems Laboratory | April 26, 2026

This notebook is a technical recreation guide for **soqv1**: a real-time audio source tracker built from a Kinect 360 sensor, an ItsyBitsy M4, a BNO055 IMU, and a servo motor. By the end of this document you should be able to reproduce every layer of the system from stract including hardware wiring, firmware, C capture program, Python signal processing, and the QML visualizer.

## 1. System Architecture

The system is a pipeline of five stages that run continuously once the hardware is powered and the software is started.

**Stage 1 — Audio capture (C program on the laptop).** The `audio_capture` binary, built from the libfreenect library, opens the Kinect over USB, configures it for four-channel audio at 16 kHz, and streams raw audio samples to standard output as a continuous binary stream.

**Stage 2 — Signal processing (Python on the laptop).** `read_kinect.py` reads that binary stream from standard input in 128-frame blocks. For each block it runs the GCC-PHAT algorithm on the two outermost microphones to estimate the direction of the sound source. The result is an angular error — how many degrees off-center the source is.

**Stage 3 — Serial communication.** The Python script sends the angular error as a plain text number over a USB serial connection at 115200 baud to the ItsyBitsy M4. The microcontroller also sends IMU heading data back up the same serial link so the Python script knows which direction the sensor is actually pointing.

**Stage 4 — Embedded control (ItsyBitsy M4).** The firmware running on the ItsyBitsy M4 reads the incoming error value and runs a PD control loop at 50 Hz. Each iteration the loop computes a correction, adds it to the current servo position, and writes the new position to the pan servo via PWM on pin 9. Separately, it polls the BNO055 IMU over I2C every 50 ms and reports the heading back over serial.

**Stage 5 — Visualization (Python on the laptop).** The Python script bundles the current angle, error, RMS level, IMU heading, and system state into a JSON packet and sends it over UDP to port 5555. The `visualizer.py` application receives these packets at roughly 60 Hz and updates the display in real time.

The physical path of the audio signal mirrors this pipeline: sound travels through air, reaches the Kinect microphones, gets digitized, processed, and turned into a servo command that physically rotates the Kinect toward the source.

## 2. Components

| # | Component | Notes |
|---|-----------|-------|
| 1 | **Xbox Kinect 360 (model 1414)** | 4-mic linear array, 0.226 m outer spacing. Available cheaply on eBay. |
| 2 | **Kinect USB adapter** | Splits the Kinect proprietary connector into USB-A + 12 V barrel jack. Required — without an Xbox 360 console the Kinect needs external power. |
| 3 | **12 V DC power supply** | Feeds the barrel jack on the Kinect adapter. |
| 4 | **Standard hobby servo (180°)** | Any servo with a standard 3-pin PWM connector. We used a common SG90-class pan servo. |
| 5 | **Pan bracket (3D-printed)** | Mounts the Kinect on the servo shaft. The blue (v2) bracket adds proper screw holes; use it over the red (v1) prototype. |
| 6 | **Adafruit ItsyBitsy M4 Express** | ATSAMD51. Runs Arduino + PID_v2 firmware. |
| 7 | **Adafruit BNO055 breakout** | 9-DOF IMU, I2C. Provides absolute heading feedback. |
| 8 | **USB cable (Micro-B or USB-C)** | ItsyBitsy M4 to laptop — power + serial. |
| 9 | **Female–female jumper wires** | Servo signal + BNO055 SDA + SCL. |
| 10 | **Breadboard / proto board** | For tidying up I2C and servo connections. |

## 3. Hardware Wiring

### 3.1 Servo to ItsyBitsy M4

The servo has three wires: signal (usually orange or yellow), power (red), and ground (brown or black).

Connect the signal wire to **pin 9** on the ItsyBitsy M4. This is the pin the firmware uses for PWM output via `panServo.attach(9)`. Connect the red power wire to the ItsyBitsy's **5V pin**, which is powered directly from the USB connection to the laptop. Connect the ground wire to any **GND pin** on the ItsyBitsy.

> **Note:** An SG90-class servo draws enough current that USB power through the ItsyBitsy is sufficient. If you use a high-torque servo, power it from a separate 5 V supply and connect that supply's ground to the ItsyBitsy GND so the two share a common reference.

### 3.2 BNO055 to ItsyBitsy M4 (I2C)

The BNO055 breakout board communicates over I2C and needs four connections.

Connect **VIN** on the BNO055 to the **3.3V pin** on the ItsyBitsy. Connect **GND** to **GND**. Connect **SDA** to the ItsyBitsy's **SDA pin** (pin 22, the hardware I2C data line). Connect **SCL** to the ItsyBitsy's **SCL pin** (pin 23, the hardware I2C clock line).

The firmware opens the BNO055 at I2C address `0x28`:
```cpp
Adafruit_BNO055 bno = Adafruit_BNO055(55, 0x28, &Wire);
```
If the ADR pin on your breakout is pulled high, change `0x28` to `0x29`.

### 3.3 Kinect to Laptop

The Kinect 360 uses a proprietary connector that combines USB data and power in a single plug. It was designed to draw power from the Xbox 360 console, so connecting it to a laptop requires a USB adapter.

Plug the Kinect's proprietary connector into the adapter. The adapter splits into two outputs: a standard USB-A plug, which connects to the laptop, and a barrel jack, which connects to a 12 V DC power supply. Once both are connected and the power supply is on, the Kinect will appear as a USB device. The `libfreenect` library handles all driver-level communication — no Xbox console or Microsoft drivers are needed.

### 3.4 Physical Assembly

Follow these steps in order before powering anything on.

1. Press the servo motor into the base of the v2 (blue) bracket and secure it with flush-head M2 screws. Make sure the screw heads sit flush with the surface and do not protrude into the motor housing.
2. Slide the Kinect into the top of the bracket and secure it so it cannot shift during rotation.
3. Mount the BNO055 on the bracket so it rotates together with the Kinect. This is critical — the heading the IMU reports must reflect the actual direction the microphone array is pointing. If the IMU is mounted on a fixed surface instead, the heading will not track the servo rotation.
4. Connect all wires before applying power.

## 4. Software Dependencies

### 4.1 System packages

**macOS (Homebrew):**
```bash
brew install libusb cmake
```

**Linux (apt):**
```bash
sudo apt install libusb-1.0-0-dev cmake build-essential
```

On Linux, add a udev rule so the Kinect is accessible without root:
```
# /etc/udev/rules.d/66-kinect.rules
SUBSYSTEM=="usb", ATTR{idVendor}=="045e", ATTR{idProduct}=="02ad", MODE="0666"
SUBSYSTEM=="usb", ATTR{idVendor}=="045e", ATTR{idProduct}=="02ae", MODE="0666"
```
Then run `sudo udevadm control --reload-rules`.

### 4.2 Arduino libraries

Install via Arduino IDE → Library Manager:
- `Adafruit BNO055`
- `Adafruit Unified Sensor` (dependency)
- `PID_v2` by br3ttb

Board: **Adafruit ItsyBitsy M4 Express** (Boards Manager → Adafruit SAMD Boards).

### 4.3 Python environment

Python 3.11+ recommended (PySide6 wheels available for 3.11–3.13).

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install numpy pyserial PySide6
```

## 5. Building libfreenect

The repo ships `libfreenect/` as a submodule. Build it in-tree:

```bash
cd libfreenect
mkdir build && cd build
cmake .. -DBUILD_EXAMPLES=ON
make -j$(nproc)
```

Two binaries land in `build/bin/`:
- **`audio_capture`** — streams raw 4-channel int32 audio to stdout
- **`measure_rms`** — prints RMS per 128-sample block to stdout; use this to calibrate the detection threshold

Verify the Kinect is detected:
```bash
./build/bin/audio_capture
# stderr: Number of devices found: 1
# stderr: Capturing audio. Press Ctrl+C to stop.
```

If you see `Number of devices found: 0`, check that the USB adapter is providing 12 V and that the USB cable is fully seated.

## 6. `audio_capture.c` — Raw Audio Streaming

`libfreenect/examples/audio_capture.c` is the bridge between the Kinect hardware and the Python processing script. It opens the sensor, registers an audio callback, and writes raw samples to stdout.

```c
#include "libfreenect.h"
#include "libfreenect_audio.h"
#include <stdio.h>
#include <stdlib.h>
#include <signal.h>

static freenect_context *f_ctx;
static freenect_device  *f_dev;
static volatile int die = 0;

void sigint_handler(int s) { die = 1; }

void in_callback(freenect_device *dev, int num_samples,
                 int32_t *mic1, int32_t *mic2,
                 int32_t *mic3, int32_t *mic4,
                 int16_t *cancelled, void *unknown) {
    int i;
    for (i = 0; i < num_samples; i++) {
        int32_t frame[4];
        frame[0] = mic1[i];
        frame[1] = mic2[i];
        frame[2] = mic3[i];
        frame[3] = mic4[i];
        fwrite(frame, sizeof(int32_t), 4, stdout);
    }
    fflush(stdout);
}

int main(int argc, char **argv) {
    signal(SIGINT, sigint_handler);
    freenect_init(&f_ctx, NULL);
    freenect_set_log_level(f_ctx, FREENECT_LOG_WARNING);
    freenect_select_subdevices(f_ctx, FREENECT_DEVICE_AUDIO);
    freenect_open_device(f_ctx, &f_dev, 0);
    freenect_set_audio_in_callback(f_dev, in_callback);
    freenect_start_audio(f_dev);
    while (!die && freenect_process_events(f_ctx) >= 0) {}
    freenect_stop_audio(f_dev);
    freenect_close_device(f_dev);
    freenect_shutdown(f_ctx);
    return 0;
}
```

**Key points:**
- The Kinect delivers audio at **16 kHz**, 32-bit signed integer per channel.
- `in_callback` fires each time a new batch of samples is ready via `freenect_process_events`.
- Each `fwrite` writes one interleaved frame of 4 × `int32_t` = **16 bytes** to stdout.
- `fflush(stdout)` after every batch keeps latency low — without it the C runtime buffer holds samples back.
- `FREENECT_DEVICE_AUDIO` activates only the audio subdevice, leaving RGB/depth cameras off.
- All log output goes to stderr so it does not corrupt the binary stdout stream.

**Wire format consumed by `read_kinect.py`:**
```
[ mic1_s32 | mic2_s32 | mic3_s32 | mic4_s32 ]  ×  num_samples
```
128 such frames = `128 × 4 channels × 4 bytes = 2048 bytes` per processing block.

## 7. `measure_rms.c` — Calibrating the Detection Threshold

Before running the full pipeline you need to know what RMS value separates silence from a real audio source in your room. `measure_rms.c` prints one RMS value per 128-sample block from mic1 to stdout so you can observe the noise floor live.

```c
#include "libfreenect.h"
#include "libfreenect_audio.h"
#include <stdio.h>
#include <stdlib.h>
#include <signal.h>
#include <math.h>

#define BLOCK_SIZE 128

static freenect_context *f_ctx;
static freenect_device  *f_dev;
static volatile int die = 0;
static int32_t buf[BLOCK_SIZE];
static int buf_idx = 0;

void sigint_handler(int s) { die = 1; }

void in_callback(freenect_device *dev, int num_samples,
                 int32_t *mic1, int32_t *mic2,
                 int32_t *mic3, int32_t *mic4,
                 int16_t *cancelled, void *unknown) {
    int i;
    for (i = 0; i < num_samples; i++) {
        buf[buf_idx++] = mic1[i];
        if (buf_idx >= BLOCK_SIZE) {
            double rms = 0.0;
            int j;
            for (j = 0; j < BLOCK_SIZE; j++) {
                double s = (double)buf[j];
                rms += s * s;
            }
            rms = sqrt(rms / BLOCK_SIZE);
            printf("%.0f\n", rms);
            fflush(stdout);
            buf_idx = 0;
        }
    }
}

int main(int argc, char **argv) {
    signal(SIGINT, sigint_handler);
    freenect_init(&f_ctx, NULL);
    freenect_set_log_level(f_ctx, FREENECT_LOG_WARNING);
    freenect_select_subdevices(f_ctx, FREENECT_DEVICE_AUDIO);
    freenect_open_device(f_ctx, &f_dev, 0);
    freenect_set_audio_in_callback(f_dev, in_callback);
    freenect_start_audio(f_dev);
    fprintf(stderr, "Measuring RMS. Ctrl+C to stop.\n");
    while (!die && freenect_process_events(f_ctx) >= 0) {}
    freenect_stop_audio(f_dev);
    freenect_close_device(f_dev);
    freenect_shutdown(f_ctx);
    return 0;
}
```

**How to use it:**
```bash
# 1. Run with the room quiet — note the idle RMS range
./build/bin/measure_rms

# 2. Play a sound source at a normal tracking distance — note the active RMS
# 3. Set THRESHOLD in read_kinect.py to a value between the two ranges
```

The project uses `THRESHOLD = 8_000_000`. In a quiet lab this gave a clean separation. Louder room → raise it; source far from Kinect → lower it.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Replace these with real values pasted from your measure_rms output.
rng = np.random.default_rng(7)
quiet_rms  = rng.normal(loc=1_500_000, scale=300_000, size=200).clip(min=0)
active_rms = rng.normal(loc=25_000_000, scale=6_000_000, size=200).clip(min=0)

THRESHOLD = 8_000_000

t_q = np.arange(len(quiet_rms))
t_a = np.arange(len(quiet_rms), len(quiet_rms) + len(active_rms))

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_q, quiet_rms / 1e6,  color='steelblue', label='Quiet room')
ax.plot(t_a, active_rms / 1e6, color='tomato',    label='Active source')
ax.axhline(THRESHOLD / 1e6, color='black', linestyle='--', linewidth=1.5,
           label=f'THRESHOLD = {THRESHOLD/1e6:.0f}M')
ax.set_xlabel('Block index  (128 samples @ 16 kHz = 8 ms each)')
ax.set_ylabel('RMS  (×10⁶)')
ax.set_title('measure_rms output — threshold calibration')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

print(f'Quiet  RMS mean : {quiet_rms.mean():.0f}')
print(f'Active RMS mean : {active_rms.mean():.0f}')
print(f'Threshold       : {THRESHOLD}  (set in read_kinect.py)')

## 8. `read_kinect.py` — GCC-PHAT, State Machine, Serial Bridge

This is the core processing script. It reads binary audio from `audio_capture` on stdin, estimates the direction of arrival, manages system state, drives the servo over serial, and sends telemetry to the visualizer over UDP.

### 8.1 Constants

```python
SAMPLE_RATE    = 16000
MIC_SPACING    = 0.226        # metres, outer mic pair (mic1 and mic4)
SPEED_OF_SOUND = 343          # m/s at ~20 C
MAX_TAU        = MIC_SPACING / SPEED_OF_SOUND   # ~0.659 ms
BLOCK_SIZE     = 128          # frames per processing block
BLOCK_BYTES    = BLOCK_SIZE * 4 * 4  # 2048 bytes

VIZ_ADDR       = ('127.0.0.1', 5555)
SERIAL_PORT    = '/dev/cu.usbmodem1301'   # change to match your ItsyBitsy port
BAUD_RATE      = 115200

THRESHOLD          = 8_000_000  # RMS below this -> IDLE
LOCK_BAND_DEG      = 5.0        # error within +/-5 deg triggers lock countdown
LOCK_HOLD_SECONDS  = 1.0        # stay in band for 1 s to reach LOCKED
IDLE_HOLD_SECONDS  = 0.5        # debounce for IDLE transitions
ERROR_EMA_ALPHA    = 0.3        # exponential moving average weight
IMU_STALE_SECONDS  = 0.75       # heading older than this is ignored
```

Find your serial port:
```bash
ls /dev/cu.*      # macOS
ls /dev/ttyACM*   # Linux
```

### 8.2 Reading audio from stdin

```python
while True:
    raw  = sys.stdin.buffer.read(BLOCK_BYTES)   # 2048 bytes = 128 interleaved frames
    if len(raw) < BLOCK_BYTES:
        break
    data = np.frombuffer(raw, dtype=np.int32).reshape(-1, 4)
    mic1 = data[:, 0].astype(np.float64)   # outer left
    mic4 = data[:, 3].astype(np.float64)   # outer right
```

Only the outer two channels are used — they give the widest baseline (0.226 m) and therefore the best angular resolution.

### 8.3 GCC-PHAT

```python
def gcc_phat(sig, refsig):
    n = len(sig) + len(refsig)                        # zero-pad to combined length
    sig_fft    = np.fft.rfft(sig,    n=n)
    refsig_fft = np.fft.rfft(refsig, n=n)
    cross_power = sig_fft * np.conj(refsig_fft)       # cross-power spectrum
    cross_power /= np.abs(cross_power) + 1e-10        # PHAT whitening (epsilon avoids div/0)
    cross_correlation = np.fft.irfft(cross_power)
    max_shift = int(SAMPLE_RATE * MAX_TAU) + 1        # ~11 samples
    cross_correlation = np.concatenate(
        (cross_correlation[-max_shift:], cross_correlation[:max_shift + 1])
    )
    shift = np.argmax(cross_correlation) - max_shift
    return shift / SAMPLE_RATE                         # TDOA in seconds
```

### 8.4 Angle estimation and filtering

```python
tau   = gcc_phat(mic1, mic4)
ratio = np.clip((tau * SPEED_OF_SOUND) / MIC_SPACING, -1, 1)

if abs(ratio) > 0.9:    # near endfire -- arcsin becomes too sensitive, skip
    continue

raw_error      = float(np.degrees(np.arcsin(ratio)))
smoothed_error = ERROR_EMA_ALPHA * raw_error + (1 - ERROR_EMA_ALPHA) * smoothed_error

last_errors.append(raw_error)
if len(last_errors) >= 3:
    recent = last_errors[-3:]
    spread = max(recent) - min(recent)
    if spread < 15:                                # consistency gate
        avg_error = sum(recent) / len(recent)
        arduino.write(f'{int(avg_error)}\n'.encode())
        last_errors.clear()
    elif len(last_errors) > 5:
        last_errors.pop(0)
```

The **0.9 ratio cutoff** rejects sources past ±64° — beyond that the arcsin gradient blows up.

The **consistency gate** (spread < 15° over last 3 measurements) suppresses single-sample spikes from reflections before they reach the servo.

### 8.5 State machine

```python
def derive_state(signal_present, measurement_valid, smoothed_error_deg, in_lock_since, now):
    if not signal_present:
        return 'IDLE', None
    if measurement_valid and abs(smoothed_error_deg) <= LOCK_BAND_DEG:
        if in_lock_since is None:
            in_lock_since = now
        if now - in_lock_since >= LOCK_HOLD_SECONDS:
            return 'LOCKED', in_lock_since
        return 'TRACKING', in_lock_since
    return 'TRACKING', None
```

| State | Entry condition | Servo command |
|-------|----------------|---------------|
| IDLE | RMS < THRESHOLD for > 0.5 s | `0\n` (hold position) |
| TRACKING | Signal present, valid measurement | error value |
| LOCKED | In ±5° band for ≥ 1 s continuously | error value |

### 8.6 IMU serial parsing

The ItsyBitsy sends heading lines at 20 Hz in the format `H:142.3,IMU:1,CAL:3`. The parser accepts both this CSV format and a JSON variant for forward compatibility. If no valid line arrives within `IMU_STALE_SECONDS = 0.75 s`, the heading is marked stale and the visualizer shows IMU STALE.

### 8.7 Running the pipeline

```bash
source .venv/bin/activate
./libfreenect/build/bin/audio_capture | python3 read_kinect.py
```

In a separate terminal, start the visualizer:
```bash
source .venv/bin/activate
python3 visualizer.py
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SAMPLE_RATE    = 16000
MIC_SPACING    = 0.226
SPEED_OF_SOUND = 343
MAX_TAU        = MIC_SPACING / SPEED_OF_SOUND

def gcc_phat(sig, refsig):
    n = len(sig) + len(refsig)
    sig_fft    = np.fft.rfft(sig,    n=n)
    refsig_fft = np.fft.rfft(refsig, n=n)
    cross_power = sig_fft * np.conj(refsig_fft)
    cross_power /= np.abs(cross_power) + 1e-10
    cc = np.fft.irfft(cross_power)
    max_shift = int(SAMPLE_RATE * MAX_TAU) + 1
    cc = np.concatenate((cc[-max_shift:], cc[:max_shift + 1]))
    shift = np.argmax(cc) - max_shift
    return shift / SAMPLE_RATE, cc, max_shift

true_angle    = 30.0
true_tau      = MIC_SPACING * np.sin(np.radians(true_angle)) / SPEED_OF_SOUND
delay_samples = int(true_tau * SAMPLE_RATE)

t   = np.arange(128) / SAMPLE_RATE
rng = np.random.default_rng(42)
tone = np.sin(2 * np.pi * 400 * t)
mic1 = tone + rng.normal(0, 0.05, 128)
mic4 = np.roll(tone, delay_samples) + rng.normal(0, 0.05, 128)

tau_est, cc, max_shift = gcc_phat(mic1, mic4)
ratio     = np.clip((tau_est * SPEED_OF_SOUND) / MIC_SPACING, -1, 1)
angle_est = np.degrees(np.arcsin(ratio))
lags      = np.arange(-max_shift, max_shift + 1)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(t * 1000, mic1, label='Mic 1 (reference)')
axes[0].plot(t * 1000, mic4, label='Mic 4 (delayed)', linestyle='--')
axes[0].set_xlabel('Time (ms)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('128-frame audio block — 400 Hz tone at 30 deg')
axes[0].legend()
axes[0].grid(True)

axes[1].plot(lags, cc, color='tab:blue')
axes[1].axvline(delay_samples,         color='red',   linestyle='--',
                label=f'True delay: {delay_samples} samples')
axes[1].axvline(tau_est * SAMPLE_RATE, color='green', linestyle=':',
                label=f'Estimate: {tau_est*SAMPLE_RATE:.1f} samples')
axes[1].axvspan(-max_shift * 0.9, max_shift * 0.9, alpha=0.07, color='orange',
                label='0.9 ratio cutoff region')
axes[1].set_xlabel('Lag (samples)')
axes[1].set_ylabel('GCC-PHAT')
axes[1].set_title(f'Cross-correlation — estimated angle: {angle_est:.1f} deg')
axes[1].legend(fontsize=8)
axes[1].grid(True)

plt.suptitle('GCC-PHAT — mirrors gcc_phat() in read_kinect.py', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'True angle       : {true_angle:.1f} deg')
print(f'Estimated angle  : {angle_est:.1f} deg')
print(f'True tau         : {true_tau*1000:.3f} ms  ({delay_samples} samples)')
print(f'Estimated tau    : {tau_est*1000:.3f} ms')
print(f'Valid range      : sources within +/-{np.degrees(np.arcsin(0.9)):.0f} deg of boresight')

## 9. `soqv1.ino` — Embedded PD Controller

Flash via Arduino IDE: board **Adafruit ItsyBitsy M4 (SAMD51)**, correct port.

### 9.1 Declarations and gains

```cpp
#include <PID_v2.h>
#include <Servo.h>
#include <Wire.h>
#include <Adafruit_BNO055.h>
#include <utility/imumaths.h>

Servo panServo;
Adafruit_BNO055 bno = Adafruit_BNO055(55, 0x28, &Wire);

double error = 0, correction = 0, setpoint = 0;
double Kp = 0.06, Ki = 0.00, Kd = 0.005;
PID_v2 panPID(Kp, Ki, Kd, PID::Direct);

double servoPos = 90;                    // start centred
const unsigned long PID_UPDATE_MS = 20;  // 50 Hz
const unsigned long IMU_REPORT_MS = 50;  // 20 Hz
```

### 9.2 Serial input — receiving the error from Python

```cpp
void readIncomingError() {
    while (Serial.available() > 0) {
        char c = (char)Serial.read();
        if (c == '\r') continue;
        if (c == '\n') {
            serialBuffer[serialIndex] = '\0';
            if (serialIndex > 0) {
                error = atof(serialBuffer) * -1;  // sign inversion matches servo orientation
                error = constrain(error, -90, 90);
            }
            serialIndex = 0;
            continue;
        }
        if (serialIndex < sizeof(serialBuffer) - 1)
            serialBuffer[serialIndex++] = c;
        else
            serialIndex = 0;  // buffer overrun -- discard and reset
    }
}
```

Python sends ASCII like `-12\n`. The `* -1` corrects for the bracket's physical orientation — if your servo turns the wrong way, remove it.

### 9.3 PD loop

```cpp
void loop() {
    unsigned long now = millis();
    readIncomingError();
    if (now - lastPidUpdateMs >= PID_UPDATE_MS) {
        lastPidUpdateMs = now;
        correction = panPID.Run(error);      // PID_v2 tracks dt internally
        servoPos  += correction;
        servoPos   = constrain(servoPos, 0, 180);
        panServo.write((int)servoPos);
    }
    reportImuHeading(now);
}
```

`panPID.SetOutputLimits(-20, 20)` in `setup()` clamps each correction to ±20°/step, limiting slew to 1000°/s max. This prevents mechanical shock and smooths motion.

### 9.4 IMU heading report

```cpp
void reportImuHeading(unsigned long now) {
    if (now - lastImuReportMs < IMU_REPORT_MS) return;
    if (Serial.availableForWrite() < 24) return;  // don't stall the PD loop
    lastImuReportMs = now;
    if (!imuReady) { Serial.println("IMU:0"); return; }
    imu::Vector<3> euler = bno.getVector(Adafruit_BNO055::VECTOR_EULER);
    uint8_t sysCal, gyroCal, accelCal, magCal;
    bno.getCalibration(&sysCal, &gyroCal, &accelCal, &magCal);
    Serial.print("H:");
    Serial.print(euler.x(), 1);
    Serial.print(",IMU:1,CAL:");
    Serial.println(sysCal);
}
```

`euler.x()` is the BNO055 Euler heading, 0–360° absolute. CAL is the system calibration score 0–3; wave the sensor in a figure-8 until it reaches 3.

### 9.5 Setup

```cpp
void setup() {
    Serial.begin(115200);
    Serial.setTimeout(1);
    panServo.attach(9);
    panServo.write(90);           // centre servo on power-up
    panPID.SetOutputLimits(-20, 20);
    panPID.Start(error, 0, setpoint);
    imuReady = bno.begin();
    if (imuReady) {
        delay(1000);
        bno.setExtCrystalUse(true);  // better accuracy with external crystal
    }
}
```

### 9.6 Gain tuning guide

| Config | Kp | Ki | Kd | Result |
|--------|----|----|-----|--------|
| **Final (chosen)** | 0.06 | 0 | 0.005 | Smooth, ~1 s settle, 0.0 deg avg steady-state error |
| With integral | 0.06 | 0.01 | 0.010 | 0.4 deg avg error, more transient oscillation |
| High derivative | 0.06 | 0 | 0.020 | Never settles — Kd amplifies GCC-PHAT noise, -16.5 deg avg error |

Start with Kp = 0.06, Ki = 0, Kd = 0 and increase Kd slowly. If the servo hunts back and forth, Kd is too high.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt     = 0.02
t      = np.arange(0, 3.0, dt)
target = 30.0

def simulate(Kp, Ki, Kd, target, t, noise_std=2.0, out_limit=20):
    pos, integral, prev_err = 90.0, 0.0, None
    errors = []
    rng = np.random.default_rng(42)
    for _ in t:
        raw_err  = (target - (pos - 90.0)) + rng.normal(0, noise_std)
        if prev_err is None:
            prev_err = raw_err
        deriv    = (raw_err - prev_err) / dt
        integral += raw_err * dt
        output   = np.clip(Kp * raw_err + Ki * integral + Kd * deriv, -out_limit, out_limit)
        pos      = np.clip(pos + output, 0, 180)
        prev_err = raw_err
        errors.append(target - (pos - 90.0))
    return np.array(errors)

configs = [
    (0.06, 0.00, 0.005, 'Kp=0.06 Kd=0.005  (final)',   'tab:blue'),
    (0.06, 0.01, 0.010, 'Kp=0.06 Ki=0.01 Kd=0.01',    'tab:orange'),
    (0.06, 0.00, 0.020, 'Kp=0.06 Kd=0.02  (rejected)', 'tab:red'),
]

fig, ax = plt.subplots(figsize=(12, 5))
ax.axhline(0, color='black', linewidth=1, linestyle='--')
ax.axhspan(-5, 5, alpha=0.1, color='green', label='+-5 deg lock band')

for Kp, Ki, Kd, label, color in configs:
    errors = simulate(Kp, Ki, Kd, target, t)
    ax.plot(t, errors, label=label, color=color)
    print(f'{label:42s}  final avg error = {np.mean(errors[-25:]):+.1f} deg')

ax.set_xlabel('Time (s)')
ax.set_ylabel('Tracking error (deg)')
ax.set_title('PD Step Response  —  30 deg step, GCC-PHAT noise sigma = 2 deg')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 10. `visualizer.py` — Real-Time QML Visualizer

Start the visualizer before or after `read_kinect.py` — it shows stale state until packets arrive.

### 10.1 UDP packet schema

`read_kinect.py` sends JSON datagrams to `127.0.0.1:5555`. Each packet:

```json
{
  "timestamp":          1745697600.123,
  "rms":                18500000.0,
  "measurement_valid":  true,
  "raw_error_deg":      -12.4,
  "smoothed_error_deg": -10.1,
  "tracker_heading_deg": 142.3,
  "imu_valid":          true,
  "imu_calibration":    3,
  "state":              "TRACKING"
}
```

### 10.2 TelemetryBridge

`TelemetryBridge(QObject)` exposes Python state to QML via `@Property` decorators. A non-blocking UDP socket is drained at ~60 Hz by a QTimer:

```python
self._poll_timer = QTimer(self)
self._poll_timer.timeout.connect(self.poll_socket)
self._poll_timer.start(16)   # ~60 Hz
```

The socket is set non-blocking so `poll_socket` drains all queued datagrams per tick without stalling the UI thread.

### 10.3 dB conversion

```python
RMS_REFERENCE = 560_000.0   # empirical scaling for this Kinect

def rms_to_relative_db(rms):
    if rms <= 0:
        return -60.0
    return clamp(20.0 * math.log10(rms / RMS_REFERENCE), -60.0, 24.0)
```

`RMS_REFERENCE` is not an absolute SPL reference — it is a scaling constant chosen so a speaking voice at ~1 m registers around 0 dBr. Adjust it to taste.

### 10.4 Audio dot smoothing

The orbiting dot is smoothed with its own EMA (`AUDIO_DOT_POSITION_EMA_ALPHA = 0.22`). Lower alpha → slower, more cinematic. Raise it toward 1.0 to snap immediately to the current heading.

### 10.5 History panel

```python
TIMELINE_SECONDS = 10.0

self._history.append({
    'timestamp': timestamp,
    'value':     float(packet['smoothed_error_deg']),
    'valid':     measurement_valid,
    'activity':  clamp((rms_to_relative_db(self._rms) + 36.0) / 42.0, 0.0, 1.0),
    'state':     state,
})
cutoff = time.time() - TIMELINE_SECONDS
self._history = [p for p in self._history if p['timestamp'] >= cutoff]
```

`activity` drives the audio energy heatmap behind the error trace in `HistoryPanel.qml`.

### 10.6 QML structure

```
ui/
├── Main.qml          -- ApplicationWindow, ColumnLayout, Esc/F shortcuts
├── HeroStage.qml     -- Large animated sensor + orbiting audio dot + propagation arcs
├── SignalPanel.qml   -- dB meter, current angle, IMU status label
└── HistoryPanel.qml  -- 10-second scrolling error timeline + energy heatmap
```

All QML files bind to the `telemetry` context property injected via:
```python
engine.rootContext().setContextProperty('telemetry', bridge)
```
Press **Esc** to quit, **F** to toggle fullscreen.

## 11. End-to-End Startup Sequence

```
1. Flash soqv1.ino to the ItsyBitsy M4.
   Verify the servo centres at 90 deg on power-up.

2. Open the Arduino Serial Monitor at 115200 baud.
   Confirm lines like:  H:142.3,IMU:1,CAL:3
   Wait for CAL to reach 3 before running the pipeline.

3. Build libfreenect (first time only):
       cd libfreenect && mkdir -p build && cd build
       cmake .. -DBUILD_EXAMPLES=ON && make -j$(nproc)

4. Calibrate the RMS threshold (first time, or new room):
       ./libfreenect/build/bin/measure_rms
   Note quiet-room values, then play your source and note active values.
   Set THRESHOLD in read_kinect.py between them.

5. Set SERIAL_PORT in read_kinect.py:
       ls /dev/cu.*      # macOS
       ls /dev/ttyACM*   # Linux

6. Terminal A:
       source .venv/bin/activate
       python3 visualizer.py

7. Terminal B:
       source .venv/bin/activate
       ./libfreenect/build/bin/audio_capture | python3 read_kinect.py

8. Make a sound. The servo pans toward it.
   The visualizer dot turns green when LOCKED (+-5 deg for >= 1 s).
```

### Troubleshooting

| Symptom | Likely cause | Fix |
|---------|-------------|-----|
| `Number of devices found: 0` | No 12 V to Kinect, or bad USB cable | Check adapter barrel jack, reseat USB |
| Servo jitters constantly | Kd too high or THRESHOLD too low | Lower Kd; raise THRESHOLD |
| Servo never moves | THRESHOLD too high or wrong serial port | Run measure_rms; check port |
| IMU STALE in visualizer | BNO055 wiring issue or CAL = 0 | Check I2C wires; wave in figure-8 |
| Dot orbits wrong direction | Servo sign or bracket orientation | Flip the `* -1` in `readIncomingError()` |
| Visualizer shows nothing | Wrong UDP port or not running | Start visualizer before pipeline |

## 12. Key Parameters — Quick Reference

| Parameter | File | Default | What to adjust it for |
|-----------|------|---------|----------------------|
| `THRESHOLD` | `read_kinect.py` | 8 000 000 | Louder/quieter room — use measure_rms |
| `ERROR_EMA_ALPHA` | `read_kinect.py` | 0.3 | Smoothness vs. responsiveness of angle estimate |
| `LOCK_BAND_DEG` | `read_kinect.py` | 5.0 | Tightness of the lock zone |
| `LOCK_HOLD_SECONDS` | `read_kinect.py` | 1.0 | Time in band before declaring LOCKED |
| `IDLE_HOLD_SECONDS` | `read_kinect.py` | 0.5 | Debounce time for IDLE transitions |
| `IMU_STALE_SECONDS` | `read_kinect.py` | 0.75 | Max age of a heading reading |
| `SERIAL_PORT` | `read_kinect.py` | `/dev/cu.usbmodem1301` | Must match your ItsyBitsy port |
| `Kp` | `soqv1.ino` | 0.06 | Proportional gain |
| `Kd` | `soqv1.ino` | 0.005 | Derivative gain — raise carefully, amplifies noise |
| `PID_UPDATE_MS` | `soqv1.ino` | 20 | PD loop period (50 Hz) |
| `SetOutputLimits(-20, 20)` | `soqv1.ino` | ±20 deg/step | Max servo slew per tick |
| `RMS_REFERENCE` | `visualizer.py` | 560 000 | dB display scaling for your Kinect |
| `AUDIO_DOT_POSITION_EMA_ALPHA` | `visualizer.py` | 0.22 | Dot animation smoothness |
| `TIMELINE_SECONDS` | `visualizer.py` | 10.0 | History panel scroll window |